# Exploração dos dados

Exploração dos quatro arquivos do TCE-RS sobre a Prefeitura de Pelotas (receita e despesa de 2024 e 2025). O objetivo é entender a estrutura e os problemas dos dados antes de escrever a pipeline. Cada seção termina com as conclusões, e a lista de problemas que a pipeline precisa resolver está no fim.

## 1. Carregando os dados

In [568]:
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

receita_2024 = pd.read_csv("../data/raw/receita_2024.csv", sep=",", decimal=".", encoding="utf-8")
receita_2025 = pd.read_csv("../data/raw/receita_2025.csv", sep=",", decimal=".", encoding="utf-8")
despesa_2024 = pd.read_csv("../data/raw/despesa_2024.csv", sep=",", decimal=".", encoding="utf-8")
despesa_2025 = pd.read_csv("../data/raw/despesa_2025.csv", sep=",", decimal=".", encoding="utf-8")

arquivos = {
    "receita_2024": receita_2024,
    "receita_2025": receita_2025,
    "despesa_2024": despesa_2024,
    "despesa_2025": despesa_2025,
}

for nome, tabela in arquivos.items():
    print(nome, tabela.shape)

receita_2024 (679, 24)
receita_2025 (672, 24)
despesa_2024 (2990, 41)
despesa_2025 (2894, 41)


In [569]:
print(list(receita_2024.columns) == list(receita_2025.columns))
print(list(despesa_2024.columns) == list(despesa_2025.columns))

True
True


Os arquivos são UTF-8 com BOM e foram lidos sem problema com `encoding="utf-8"`. Os dois anos de cada tipo têm as mesmas colunas: 24 na receita e 41 na despesa.

## 2. Receita

### 2.1 Receita 2025: `NIVEL` e `NR_NIVEL`

#### Visão geral

Anotações sobre a exploração do arquivo receita_2025.csv:
- Ela tem 24 colunas e 672 linhas
- Algumas colunas, como 'NOME_UNIDADE' e 'TP_UNIDADE', têm apenas 452 linhas preenchidas, enquanto outras têm todas as linhas preenchidas
- 'NM_RECURSO' tem zero linhas preenchidas
- A coluna 'NIVEL' tem apenas dados como 'S' ou 'A'. Sendo 452 linhas com A e 220 linhas com S
- A coluna 'NR_NIVEL' tem apenas números de 1 a 9. E a maioria das linhas tem dados entre 7 e 9
- As linhas com 'NIVEL' = A possuem 'NOME_UNIDADE', enquanto as com 'NIVEL' = S têm 'NOME_UNIDADE' vazio

In [570]:
receita_2025.info()

<class 'pandas.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ANO                      672 non-null    int64  
 1   MES                      672 non-null    int64  
 2   COD_MUNICIPIO            672 non-null    int64  
 3   NOME_MUNICIPIO           672 non-null    str    
 4   CD_ORGAO                 672 non-null    int64  
 5   NOME_ORGAO               672 non-null    str    
 6   CD_RECEBIMENTO           672 non-null    int64  
 7   CD_ORGAO_ORCAMENTARIO    672 non-null    int64  
 8   NOME_ORGAO_ORCAMENTARIO  452 non-null    str    
 9   CD_UNIDADE_ORCAMENTARIA  672 non-null    int64  
 10  NOME_UNIDADE             452 non-null    str    
 11  TP_UNIDADE               452 non-null    str    
 12  CD_CONTA_SG              672 non-null    int64  
 13  DS_CONTA_SG              672 non-null    str    
 14  NIVEL                    672 non-null

In [571]:
filtro_nivel_a = receita_2025["NIVEL"] == "A"
filtro_nivel_s = receita_2025["NIVEL"] == "S"
filtro_nr_nivel_1 = receita_2025["NR_NIVEL"] == 1
filtro_nr_nivel_2 = receita_2025["NR_NIVEL"] == 2
filtro_nr_nivel_3 = receita_2025["NR_NIVEL"] == 3
filtro_nr_nivel_4 = receita_2025["NR_NIVEL"] == 4
filtro_nr_nivel_5 = receita_2025["NR_NIVEL"] == 5
filtro_nr_nivel_6 = receita_2025["NR_NIVEL"] == 6

# código da conta como texto, para usar startswith (a coluna original continua intacta)
receita_2025["CODIGO"] = receita_2025["CD_CONTA_SG"].astype(str)

receita_2025["VL_ARRECADADO"].sum()

np.float64(10455427845.77)

O valor arrecadado de todo o dataframe é R$ 10,455,427,845.77. Mais de seis vezes o valor apenas com NIVEL igual A (R$ 1,549,096,934.03), então somar tudo conta o mesmo dinheiro várias vezes.

In [572]:
receita_2025[filtro_nr_nivel_1][["DS_CONTA_SG", "CD_CONTA_SG"]]

,DS_CONTA_SG,CD_CONTA_SG
0,Receitas Correntes,100000000000000
559,Receitas de Capital,200000000000000


In [573]:
print(receita_2025[filtro_nr_nivel_1]["VL_ARRECADADO"].sum())
print(receita_2025[filtro_nr_nivel_2]["VL_ARRECADADO"].sum())
print(receita_2025[filtro_nr_nivel_3]["VL_ARRECADADO"].sum())
print(receita_2025[filtro_nr_nivel_4]["VL_ARRECADADO"].sum())

1549096934.03
1549096934.03
1549096934.0300004
1549096934.03


Cada NR_NIVEL, do 1 ao 4, soma exatamente o mesmo valor arrecadado total (R$ 1,549,096,934.03, o mesmo das linhas com NIVEL igual A). Isso provavelmente significa que as linhas se repetem e tem a ver com a decomposição da categoria em níveis.

In [574]:
pd.crosstab(receita_2025["NR_NIVEL"], receita_2025["NIVEL"])

NIVEL,A,S
NR_NIVEL,,
1,0,2
2,0,9
3,0,20
4,0,39
5,0,67
6,0,24
7,56,54
8,278,5
9,118,0


Do NR_NIVEL 1 ao 6 só existem linhas S. A partir do 7, as linhas A e S se misturam.

Conclusão: NR_NIVEL é o nível da hierarquia. NIVEL A é a ponta da hierarquia, e NIVEL S agrupa as linhas de baixo.

### 2.2 Teste: a linha S é a soma das linhas A abaixo dela

Para uma linha S, procuro as linhas A cujo `CD_CONTA_SG` começa com o mesmo prefixo (o código sem os zeros do fim). A função `testa_s_contra_a` faz esse teste para qualquer linha S de qualquer tabela.

In [575]:
def testa_s_contra_a(tabela: pd.DataFrame, linha: int) -> None:
    """Confere se uma linha S é a soma das linhas A abaixo dela."""

    if tabela.loc[linha, "NIVEL"] == "S":

        valor_s = tabela.loc[linha]["VL_ARRECADADO"]
        codigo = tabela.loc[linha]["CD_CONTA_SG"]
        prefixo = str(codigo).rstrip("0")
        filtro_nivel_a = tabela["NIVEL"] == "A"

        filtro_prefixo = tabela["CD_CONTA_SG"].astype(str).str.startswith(prefixo)
        df_filhas = tabela[filtro_prefixo & filtro_nivel_a]
        valor_df_filhas = df_filhas["VL_ARRECADADO"].sum()
        print(f"Codigo: {prefixo}")
        print(f"Total de linhas filhas: {len(df_filhas)}")
        print(f"Total de VL_ARRECADADO das linhas filhas: {valor_df_filhas:,.2f}")
        print(f"VL_ARRECADADO da linha S: {valor_s:,.2f}")

        if valor_s.round(2) == valor_df_filhas.round(2):
            print("A linha S é a soma das linhas A abaixo dela!")
        else:
            print("A linha S NÃO é a soma das linhas A abaixo dela!")
    else:
        print("A linha deve ter NIVEL S")

In [576]:
testa_s_contra_a(receita_2025, 3)
testa_s_contra_a(receita_2025, 22)

Codigo: 1112
Total de linhas filhas: 10
Total de VL_ARRECADADO das linhas filhas: 229,995,095.65
VL_ARRECADADO da linha S: 229,995,095.65
A linha S é a soma das linhas A abaixo dela!
Codigo: 1114
Total de linhas filhas: 5
Total de VL_ARRECADADO das linhas filhas: 140,985,949.00
VL_ARRECADADO da linha S: 140,985,949.00
A linha S é a soma das linhas A abaixo dela!


A linha 3 (Impostos sobre o Patrimônio, NIVEL S) arrecadou 229,995,095.65. As 10 linhas A cujo código começa com 1112 somam o mesmo valor. A linha S é o total das linhas A abaixo dela.

### 2.3 Tamanho do prefixo por nível

In [577]:
receita_2025["caracteres_codigo"] = receita_2025["CD_CONTA_SG"].astype(str).str.rstrip("0").str.len()
pd.crosstab(receita_2025["NR_NIVEL"], receita_2025["caracteres_codigo"])

caracteres_codigo,1,2,3,4,5,6,7,8,10,14,15
NR_NIVEL,,,,,,,,,,,
1,2,0,0,0,0,0,0,0,0,0,0
2,0,9,0,0,0,0,0,0,0,0,0
3,0,0,20,0,0,0,0,0,0,0,0
4,0,0,0,39,0,0,0,0,0,0,0
5,0,0,0,0,15,52,0,0,0,0,0
6,0,0,0,0,0,0,24,0,0,0,0
7,0,0,0,0,0,0,0,54,0,0,56
8,0,0,0,0,0,0,0,0,5,0,278
9,0,0,0,0,0,0,0,0,0,5,113


### 2.4 O ramo do IPTU

Partindo de uma linha A do IPTU (`111250010000001`), o ramo sobe pelas linhas S cujo código é o começo do código da ponta, um nível de cada vez.

In [578]:
# Nível 1
codigo_folha = "111250010000001"
colunas_ramo = ["NR_NIVEL", "CODIGO", "DS_CONTA_SG", "VL_ARRECADADO"]
codigo = codigo_folha[:1]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
receita_2025[(filtro_nivel_s) & (filtro_nr_nivel_1) & (filtro_codigo)][colunas_ramo]

,NR_NIVEL,CODIGO,DS_CONTA_SG,VL_ARRECADADO
0,1,100000000000000,Receitas Correntes,"1,496,486,811.37"


In [579]:
# Nível 2
codigo = codigo_folha[:2]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
receita_2025[(filtro_nivel_s) & (filtro_nr_nivel_2) & (filtro_codigo)][colunas_ramo]

,NR_NIVEL,CODIGO,DS_CONTA_SG,VL_ARRECADADO
1,2,110000000000000,"Impostos, Taxas e Contribuições de Melhoria","441,471,354.68"


In [580]:
# Nível 3
codigo = codigo_folha[:3]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
receita_2025[(filtro_nivel_s) & (filtro_nr_nivel_3) & (filtro_codigo)][colunas_ramo]

,NR_NIVEL,CODIGO,DS_CONTA_SG,VL_ARRECADADO
2,3,111000000000000,Impostos,"433,595,415.61"


In [581]:
# Nível 4
codigo = codigo_folha[:4]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
receita_2025[(filtro_nivel_s) & (filtro_nr_nivel_4) & (filtro_codigo)][colunas_ramo]

,NR_NIVEL,CODIGO,DS_CONTA_SG,VL_ARRECADADO
3,4,111200000000000,Impostos sobre o Patrimônio,"229,995,095.65"


In [582]:
# Nível 5
codigo = codigo_folha[:6]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
receita_2025[(filtro_nivel_s) & (filtro_nr_nivel_5) & (filtro_codigo)][colunas_ramo]

,NR_NIVEL,CODIGO,DS_CONTA_SG,VL_ARRECADADO
4,5,111250000000000,Imposto sobre a Propriedade Predial e Territor...,"184,095,028.53"


In [583]:
# Nível 6 (o ramo não tem linha S neste nível)
codigo = codigo_folha[:7]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
receita_2025[(filtro_nivel_s) & (filtro_nr_nivel_6) & (filtro_codigo)][colunas_ramo]

,NR_NIVEL,CODIGO,DS_CONTA_SG,VL_ARRECADADO


In [584]:
# Linhas A (a ponta do ramo)
codigo = codigo_folha[:6]
filtro_codigo = receita_2025["CODIGO"].str.startswith(codigo)
print(receita_2025[(filtro_nivel_a) & (filtro_codigo)][["VL_ARRECADADO", "NIVEL", "NR_NIVEL"]])
print(receita_2025[(filtro_nivel_a) & (filtro_codigo)]["VL_ARRECADADO"].sum().round(2))

    VL_ARRECADADO NIVEL  NR_NIVEL
5  148,422,688.20     A         7
6            0.06     A         7
7            0.00     A         7
8    2,575,923.48     A         7
9   22,059,179.86     A         7
10       1,398.00     A         7
11  11,035,838.93     A         7
184095028.53


#### O que está comprovado na receita 2025

- A receita é uma hierarquia de categorias e subcategorias. O NR_NIVEL é a profundidade, e o CD_CONTA_SG mostra a ligação: o código do filho começa com o código do pai (o ramo do IPTU, do nível 1 ao 5, comprova).
- NIVEL = S agrupa, NIVEL = A é a ponta. A linha S é a soma das linhas A abaixo dela (confirmado nas linhas testadas).
- O que cada NIVEL significa. A (analítica) é a ponta da hierarquia: é onde o valor é registrado, e as linhas A só aparecem do NR_NIVEL 7 ao 9. S (sintética) é uma categoria que agrupa as de baixo, e vai do NR_NIVEL 1 ao 8. Exemplo: a linha S do IPTU (nível 5) arrecadou R$ 184.095.028,53, e as 7 linhas A sob ela somam exatamente esse valor.
- Somar a tabela inteira conta o mesmo dinheiro várias vezes. Para o total real, some só NIVEL == "A". Isso vai direto para a Etapa 5.
- O tamanho do prefixo depende do nível (1, 2, 3, 4, 6, 7, 8, 10 caracteres), e o rstrip("0") erra no nível 5, por cortar um zero legítimo.
- A hierarquia pula níveis. O IPTU vai do nível 5 direto para a ponta no nível 7.

### 2.5 Receita 2024

In [585]:
print(receita_2024.shape)
receita_2024.info()

(679, 24)
<class 'pandas.DataFrame'>
RangeIndex: 679 entries, 0 to 678
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ANO                      679 non-null    int64  
 1   MES                      679 non-null    int64  
 2   COD_MUNICIPIO            679 non-null    int64  
 3   NOME_MUNICIPIO           679 non-null    str    
 4   CD_ORGAO                 679 non-null    int64  
 5   NOME_ORGAO               679 non-null    str    
 6   CD_RECEBIMENTO           679 non-null    int64  
 7   CD_ORGAO_ORCAMENTARIO    679 non-null    int64  
 8   NOME_ORGAO_ORCAMENTARIO  441 non-null    str    
 9   CD_UNIDADE_ORCAMENTARIA  679 non-null    int64  
 10  NOME_UNIDADE             441 non-null    str    
 11  TP_UNIDADE               441 non-null    str    
 12  CD_CONTA_SG              679 non-null    int64  
 13  DS_CONTA_SG              679 non-null    str    
 14  NIVEL                    67

In [586]:
receita_2024.value_counts("NIVEL")

NIVEL
A    441
S    238
Name: count, dtype: int64

In [587]:
filtro_nivel_a_2024 = receita_2024["NIVEL"] == "A"
filtro_nivel_s_2024 = receita_2024["NIVEL"] == "S"
filtro_nr_nivel_1_2024 = receita_2024["NR_NIVEL"] == 1

print(receita_2024["VL_ARRECADADO"].sum().round(2))
print(receita_2024[filtro_nivel_a_2024]["VL_ARRECADADO"].sum().round(2))
print(receita_2024[filtro_nr_nivel_1_2024]["VL_ARRECADADO"].sum().round(2))

10044803801.06
1458681677.02
1458681677.02


In [588]:
pd.crosstab(receita_2024["NR_NIVEL"], receita_2024["NIVEL"])

NIVEL,A,S
NR_NIVEL,,
1,0,3
2,0,9
3,0,21
4,0,40
5,0,75
6,0,25
7,51,59
8,285,6
9,105,0


In [589]:
receita_2024[filtro_nr_nivel_1_2024][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
0,100000000000000,Receitas Correntes,S,1,"1,472,878,245.72"
543,200000000000000,Receitas de Capital,S,1,"59,254,414.61"
665,9000000000000000,(R) Deduções da Receita Corrente,S,1,"-73,450,983.31"


In [590]:
receita_2024["caracteres_codigo"] = receita_2024["CD_CONTA_SG"].astype(str).str.rstrip("0").str.len()
pd.crosstab(receita_2024["NR_NIVEL"], receita_2024["caracteres_codigo"])

caracteres_codigo,1,2,3,4,5,6,7,8,10,14,15,16
NR_NIVEL,,,,,,,,,,,,
1,3,0,0,0,0,0,0,0,0,0,0,0
2,0,9,0,0,0,0,0,0,0,0,0,0
3,0,0,21,0,0,0,0,0,0,0,0,0
4,0,0,0,40,0,0,0,0,0,0,0,0
5,0,0,0,0,17,58,0,0,0,0,0,0
6,0,0,0,0,0,0,25,0,0,0,0,0
7,0,0,0,0,0,0,0,59,0,0,51,0
8,0,0,0,0,0,0,0,0,6,0,275,10
9,0,0,0,0,0,0,0,0,0,5,100,0


In [591]:
filtro_caracteres_codigo = receita_2024["caracteres_codigo"] == 16
receita_2024[filtro_caracteres_codigo][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
668,9111250010000101,(R) Deduções - IPTU - Principal - Renúncia Fiscal,A,8,0.00
669,9111253010000101,(R) Deduções - ITBI - Principal Renúncia Fiscal,A,8,0.00
670,9111451110000101,(R) Deduções - ISS - Principal - Renúncia Fiscal,A,8,0.00
671,9112201010000101,(R) Deduções - Taxas pela Prestação de Serviço...,A,8,0.00
673,9171151110000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-26,429,839.66"
674,9171152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-265,020.72"
675,9172150010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-30,793,411.64"
676,9172151010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-15,157,400.20"
677,9172152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-389,407.12"
678,9172953010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-415,903.97"


In [592]:
testa_s_contra_a(receita_2024, 192)
testa_s_contra_a(receita_2024, 78)
testa_s_contra_a(receita_2024, 154)

Codigo: 171
Total de linhas filhas: 104
Total de VL_ARRECADADO das linhas filhas: 488,160,843.26
VL_ARRECADADO da linha S: 488,160,843.26
A linha S é a soma das linhas A abaixo dela!
Codigo: 131102
Total de linhas filhas: 2
Total de VL_ARRECADADO das linhas filhas: 2,070,159.60
VL_ARRECADADO da linha S: 2,070,159.60
A linha S é a soma das linhas A abaixo dela!
Codigo: 1321010102
Total de linhas filhas: 11
Total de VL_ARRECADADO das linhas filhas: 1,362,586.78
VL_ARRECADADO da linha S: 1,362,586.78
A linha S é a soma das linhas A abaixo dela!


#### Conclusões da receita 2024

- A estrutura é a mesma de 2025. As linhas A ficam do NR_NIVEL 7 ao 9 e as S do 1 ao 8. Somar a tabela toda dá R$ 10,04 bi, quase sete vezes o valor das linhas A (R$ 1.458.681.677,02), então somar tudo conta o mesmo dinheiro várias vezes. O NR_NIVEL 1 soma exatamente o mesmo que as linhas A.
- Os prefixos têm os mesmos tamanhos por nível de 2025, e o nível 5 também se divide em 5 e 6 caracteres.
- O teste "a linha S é a soma das linhas A abaixo dela" bateu nas 3 linhas testadas (níveis 3, 5 e 8).

**Diferenças em relação a 2025**

- Em 2024 existe uma terceira linha no NR_NIVEL 1: "(R) Deduções da Receita Corrente" (código 9000000000000000, R$ -73.450.983,31). Em 2025 há só duas.
- Os 14 códigos que começam com 9 pertencem às deduções e têm 16 dígitos, em vez de 15 (renúncia fiscal de IPTU, ITBI, ISS e taxas, e deduções para formação do FUNDEB).
- O total das linhas A de 2024 já inclui essas deduções. Sem elas seria R$ 1.532.132.660,33.

**Problemas para a pipeline resolver**

- O `CD_CONTA_SG` deve ser lido como texto, porque o código tem 15 ou 16 dígitos.
- Decidir se a receita arrecadada considera as deduções (líquida) ou não (bruta).

### 2.6 Valores negativos

In [593]:
filtro_negativo = receita_2024["VL_ARRECADADO"] < 0
receita_2024[filtro_negativo][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
665,9000000000000000,(R) Deduções da Receita Corrente,S,1,"-73,450,983.31"
666,9100000000000000,(R) Deduções da Receita Corrente,S,2,"-73,450,983.31"
672,9170000000000000,(R) Deduções da Receita Corrente,S,3,"-73,450,983.31"
673,9171151110000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-26,429,839.66"
674,9171152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-265,020.72"
675,9172150010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-30,793,411.64"
676,9172151010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-15,157,400.20"
677,9172152010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-389,407.12"
678,9172953010000101,(R) Dedução da Receita para Formação do FUNDEB...,A,8,"-415,903.97"


In [594]:
filtro_negativo = receita_2025["VL_ARRECADADO"] < 0
receita_2025[filtro_negativo][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
216,171151110000001,Cota-Parte do Fundo de Participação dos Municí...,A,7,"-28,695,886.97"
220,171152010000001,Cota-Parte do Imposto Sobre a Propriedade Terr...,A,7,"-266,813.13"
367,172150010000001,Cota-Parte do ICMS - Principal,A,7,"-33,430,718.82"
370,172151010000001,Cota-Parte do IPVA - Principal,A,7,"-16,166,711.13"
373,172152010000001,Cota-Parte do IPI - Municípios - Principal,A,7,"-390,540.13"


In [595]:
filtro_171151110000001 = receita_2025["CD_CONTA_SG"] == 171151110000001
receita_2025[filtro_171151110000001][["CD_CONTA_SG", "DS_CONTA_SG", "NIVEL", "NR_NIVEL", "VL_ARRECADADO"]]

,CD_CONTA_SG,DS_CONTA_SG,NIVEL,NR_NIVEL,VL_ARRECADADO
215,171151110000001,Cota-Parte do Fundo de Participação dos Municí...,A,7,"143,456,215.81"
216,171151110000001,Cota-Parte do Fundo de Participação dos Municí...,A,7,"-28,695,886.97"


#### Conclusões sobre os valores negativos

Há 9 linhas com `VL_ARRECADADO` negativo em 2024 (6 A e 3 S) e 5 linhas em 2025 (todas A). Não são erro de preenchimento: são deduções da receita. O município recebe repasses como FPM, ITR, ICMS, IPVA e IPI, e 20% deles são retidos para formar o FUNDEB, o fundo da educação básica. A contabilidade registra essa retenção como valor negativo, que reduz a receita arrecadada.

- Em 2024, as deduções ficam num ramo separado, com códigos que começam com 9 e descrição "(R) Dedução...". Somam -R$ 73.450.983,31.
- Em 2025, os mesmos cinco repasses aparecem negativos, com valores parecidos, mas com o código da própria conta de receita (por exemplo, `171151110000001`, Cota-Parte do FPM) e sem a palavra "dedução" na descrição. Somam -R$ 78.950.670,18. Cada repasse aparece em duas linhas com o mesmo código: a bruta e a dedução de 20% (no FPM, R$ 143 milhões positivos e R$ 28 milhões negativos).

**Conclusões**

- Os negativos são válidos. A pipeline não deve descartá-los nem trocar o sinal.
- O total das linhas A é receita líquida nos dois anos, porque as deduções entram na soma em 2024 e em 2025.
- Comparar categorias entre os anos exige cuidado: em 2024 o desconto do FUNDEB está num ramo à parte, e em 2025 ele está misturado nas contas dos repasses.

### 2.7 Tipo do `CD_CONTA_SG`

In [596]:
receita_2025["CD_CONTA_SG"]

0      100000000000000
1      110000000000000
2      111000000000000
3      111200000000000
4      111250000000000
            ...       
667    242299010600001
668    242299011000001
669    242299011500001
670    242299011700001
671    242299011800001
Name: CD_CONTA_SG, Length: 672, dtype: int64

#### Conclusão sobre o tipo do CD_CONTA_SG

O pandas leu `CD_CONTA_SG` como `int64`. Para a pipeline, o ideal é lê-lo como texto. O código identifica uma conta, não é uma quantidade: nunca é somado, e o trabalho com ele depende de operações de texto (`startswith`, fatiamento do prefixo). Em 2024 ele ainda tem códigos de 15 e de 16 dígitos.

Nos testes de prefixo deste notebook, usei a coluna `CODIGO`, uma cópia em texto de `CD_CONTA_SG`. A coluna original ficou intacta.

## 3. Despesa

### 3.1 Visão geral

In [597]:
print(despesa_2025.shape)
despesa_2025.info()

(2894, 41)
<class 'pandas.DataFrame'>
RangeIndex: 2894 entries, 0 to 2893
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ANO                          2894 non-null   int64  
 1   MES                          2894 non-null   int64  
 2   COD_MUNICIPIO                2894 non-null   int64  
 3   NOME_MUNICIPIO               2894 non-null   str    
 4   CD_ORGAO                     2894 non-null   int64  
 5   NOME_ORGAO                   2894 non-null   str    
 6   CD_RECEBIMENTO               2894 non-null   int64  
 7   CD_ORGAO_ORCAMENTARIO        2894 non-null   int64  
 8   NOME_ORGAO_ORCAMENTARIO      2894 non-null   str    
 9   CD_UNIDADE_ORCAMENTARIA      2894 non-null   int64  
 10  NOME_UNIDADE                 2894 non-null   str    
 11  TP_UNIDADE                   2894 non-null   str    
 12  CD_FUNCAO                    2894 non-null   int64  
 13  DS_FUNCAO         

In [598]:
print(despesa_2024.shape)
despesa_2024.info()

(2990, 41)
<class 'pandas.DataFrame'>
RangeIndex: 2990 entries, 0 to 2989
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ANO                          2990 non-null   int64  
 1   MES                          2990 non-null   int64  
 2   COD_MUNICIPIO                2990 non-null   int64  
 3   NOME_MUNICIPIO               2990 non-null   str    
 4   CD_ORGAO                     2990 non-null   int64  
 5   NOME_ORGAO                   2990 non-null   str    
 6   CD_RECEBIMENTO               2990 non-null   int64  
 7   CD_ORGAO_ORCAMENTARIO        2990 non-null   int64  
 8   NOME_ORGAO_ORCAMENTARIO      2990 non-null   str    
 9   CD_UNIDADE_ORCAMENTARIA      2990 non-null   int64  
 10  NOME_UNIDADE                 2990 non-null   str    
 11  TP_UNIDADE                   2990 non-null   str    
 12  CD_FUNCAO                    2990 non-null   int64  
 13  DS_FUNCAO         

### 3.2 `DS_FUNCAO`

In [599]:
despesa_2025["DS_FUNCAO"].unique()

<ArrowStringArray>
[       'Administração',    'Segurança Pública',   'Assistência Social',
                'Saúde',             'Trabalho',             'Educação',
              'Cultura',            'Urbanismo',            'Habitação',
     'Gestão Ambiental', 'Ciência e Tecnologia',          'Agricultura',
  'Comércio e Serviços',              'Energia',           'Transporte',
     'Desporto e Lazer',   'Encargos Especiais',             'Reservas']
Length: 18, dtype: str

In [600]:
despesa_2024["DS_FUNCAO"].unique()

<ArrowStringArray>
[          'ADMINISTRACAO',       'SEGURANCA PUBLICA',
      'ASSISTENCIA SOCIAL',                   'SAUDE',
                'TRABALHO',                'EDUCACAO',
                 'CULTURA',               'URBANISMO',
               'HABITACAO',        'GESTAO AMBIENTAL',
    'CIENCIA E TECNOLOGIA',             'AGRICULTURA',
     'COMERCIO E SERVICOS',                 'ENERGIA',
              'TRANSPORTE',        'DESPORTO E LAZER',
      'ENCARGOS ESPECIAIS', 'RESERVA DE CONTINGENCIA']
Length: 18, dtype: str

#### Conclusão: DS_FUNCAO, 2024 contra 2025

Os dois anos têm as mesmas 18 funções de governo (Administração, Saúde, Educação, Assistência Social etc.), e o `CD_FUNCAO` é o mesmo nos dois (10 é saúde, 12 é educação, 99 é reservas). Mas a escrita da descrição muda:

- Em 2024 as descrições vêm em maiúsculas e sem acento (`SAUDE`, `EDUCACAO`, `ASSISTENCIA SOCIAL`). Em 2025 vêm com acento e só a inicial maiúscula (`Saúde`, `Educação`, `Assistência Social`).
- Uma função muda de nome, e não só de grafia: `RESERVA DE CONTINGENCIA` em 2024 e `Reservas` em 2025.

**Conclusão:** juntar os dois anos sem tratar essa coluna duplica as funções (`SAUDE` e `Saúde` viram duas categorias). A pipeline precisa padronizar a descrição, e a função de reservas exige um mapeamento explícito entre os dois nomes. O `CD_FUNCAO` é uma chave mais confiável que a descrição.

### 3.3 `VL_EMPENHADO`, `VL_LIQUIDADO` e `VL_PAGO`

In [601]:
def busca_valor_negativo(tabela: pd.DataFrame, coluna: str) -> float:
    """Recebe dataframe e coluna. Retorna saldo de valores negativos da coluna"""
    filtro_negativo = tabela[coluna] < 0
    df_negativos = tabela[filtro_negativo]
    saldo = df_negativos[coluna].sum()
    if saldo < 0:
        return saldo
    else:
        print(f"A coluna {coluna} não tem valores negativos")
        return 0.0


busca_valor_negativo(despesa_2025, "VL_TRANSFERENCIA")
busca_valor_negativo(despesa_2025, "VL_LIQUIDADO")
busca_valor_negativo(despesa_2025, "VL_PAGO")
busca_valor_negativo(despesa_2024, "VL_TRANSFERENCIA")
busca_valor_negativo(despesa_2024, "VL_LIQUIDADO")
busca_valor_negativo(despesa_2024, "VL_PAGO")

A coluna VL_TRANSFERENCIA não tem valores negativos
A coluna VL_LIQUIDADO não tem valores negativos
A coluna VL_PAGO não tem valores negativos
A coluna VL_TRANSFERENCIA não tem valores negativos
A coluna VL_LIQUIDADO não tem valores negativos
A coluna VL_PAGO não tem valores negativos


0.0

In [602]:
for nome, tabela in {"despesa_2024": despesa_2024, "despesa_2025": despesa_2025}.items():
    print(nome)
    print("  pago maior que liquidado:", len(tabela[tabela["VL_PAGO"] > tabela["VL_LIQUIDADO"]]))
    print("  liquidado maior que empenhado:", len(tabela[tabela["VL_LIQUIDADO"] > tabela["VL_EMPENHADO"]]))

despesa_2024
  pago maior que liquidado: 0
  liquidado maior que empenhado: 0
despesa_2025
  pago maior que liquidado: 0
  liquidado maior que empenhado: 0


In [603]:
def total_linhas_VL_EMPENHADO_igual_zero(tabela: pd.DataFrame) -> None:
    total_linhas = len(tabela)
    filtro = tabela["VL_EMPENHADO"] == 0
    linhas_iguais_zero = len(tabela[filtro])
    print(f"A tabela tem {total_linhas} linhas com {linhas_iguais_zero} linhas iguais a zero")


total_linhas_VL_EMPENHADO_igual_zero(despesa_2025)
total_linhas_VL_EMPENHADO_igual_zero(despesa_2024)

A tabela tem 2894 linhas com 1363 linhas iguais a zero
A tabela tem 2990 linhas com 1493 linhas iguais a zero


#### Conclusão: empenhado, liquidado e pago

Empenho é a reserva do dinheiro no orçamento para uma despesa, liquidação é o reconhecimento de que o serviço ou bem foi entregue, e pagamento é a saída do dinheiro. Por isso existe uma ordem: empenhado ≥ liquidado ≥ pago.

Conferi em todas as linhas, nos dois anos: não há nenhuma com `VL_PAGO` maior que `VL_LIQUIDADO`, nem com `VL_LIQUIDADO` maior que `VL_EMPENHADO`. Nenhuma coluna `VL_` da despesa tem valor negativo.

- **Empenhado maior que pago:** 983 linhas em 2024 e 954 em 2025. É a despesa assumida e ainda não paga (o "empenhado não pago").
- **Linhas com `VL_EMPENHADO == 0`:** 1.493 de 2.990 em 2024 e 1.363 de 2.894 em 2025, perto de metade da tabela. A maioria tem `VL_DOTACAO_INICIAL` preenchida: é verba prevista no orçamento que não teve gasto no ano. Não são erro, e não devem ser descartadas, porque são parte do "orçado x executado".
- **Todas as linhas são do mês 12.** Os valores são o acumulado do ano até dezembro, e não mensais.

**Conclusão para a pipeline:** as somas não mudam com as linhas sem gasto, mas contagens e médias por linha ficam distorcidas. As colunas `CD_RECURSO` e `NM_RECURSO` vêm 100% vazias e podem ser descartadas.

## 4. Problemas que a pipeline precisa resolver

**Receita**

1. **A tabela mistura linhas agrupadoras (S) e linhas da ponta (A).** Somar `VL_ARRECADADO` da tabela inteira conta o mesmo dinheiro várias vezes (cerca de sete vezes o valor real). A pipeline deve somar só `NIVEL == "A"`.
2. **`CD_CONTA_SG` é lido como número.** É um código de 15 ou 16 dígitos e deve ser lido como texto.
3. **As deduções do FUNDEB são registradas de forma diferente em cada ano.** Em 2024 estão num ramo à parte, com códigos que começam com 9 (16 dígitos) e uma terceira raiz no `NR_NIVEL` 1. Em 2025 são linhas negativas com o código da própria conta de receita. Os negativos são válidos, não podem ser descartados, e é preciso decidir se a receita considera as deduções (líquida) ou não (bruta). A soma das linhas A é líquida nos dois anos.
4. **O código da conta não identifica uma linha.** O mesmo `CD_CONTA_SG` aparece em mais de uma linha (o IPTU em três, cada repasse do FUNDEB em duas, por unidade e por `CAR_PECULIAR`). Para ter um valor por conta é preciso agrupar.
5. **A hierarquia só pode ser reconstruída pelo prefixo do código, e com cuidado.** O tamanho do prefixo muda por nível e o nível 5 tem códigos com zero legítimo no meio. Isso não afeta a soma por `NIVEL == "A"`, mas importa se for preciso agrupar por categoria.

**Despesa**

6. **`DS_FUNCAO` muda de escrita entre os anos** (maiúsculas sem acento em 2024, acentuadas em 2025) e a função de reservas muda de nome (`RESERVA DE CONTINGENCIA` e `Reservas`). É preciso padronizar, usando o `CD_FUNCAO` como chave.
7. **Cerca de metade das linhas tem `VL_EMPENHADO == 0`.** São verbas previstas sem gasto. Devem ser mantidas, mas distorcem contagens e médias.
8. **Há despesa assumida e não paga** (`VL_EMPENHADO` maior que `VL_PAGO` em cerca de um terço das linhas). É a base da pergunta "empenhado não pago".

**Nos dois**

9. **Os valores são o acumulado do ano até dezembro** (`MES = 12`). Não existe série mensal.
10. **Colunas 100% vazias:** `NM_RECURSO` na receita e `CD_RECURSO` e `NM_RECURSO` na despesa. Podem ser descartadas.
11. **Encoding:** os seis arquivos são UTF-8 com BOM. O pandas lê sem problema com `encoding="utf-8"`.